In [1]:
using NeuroDSL

# cross-entropy

In [2]:
using NeuroDSL

dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:test)

logits = randn(Float32, 4, 10)
labels = UInt8.([1, 5, 8, 2])

NeuroDSL.set!(g, :logits, logits; namespace=:test)
NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=:test)
NeuroDSL.set!(g, :loss, fill(0.0f0, 1, 1); namespace=:test)
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [:logits, :labels], :cross_entropy; namespace=:test))

ctx = NeuroDSL.CtxStore()
NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:test)
println("Loss = ", g.nodes[:test][:loss].value[1])

NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:test)
grad = g.nodes[:test][:logits].gradient
println("Gradient shape: ", size(grad))
println("Gradient sum ≈ 0 ? ", round(sum(Array(grad)), digits=6))

Loss = 1.8267962
Gradient shape: (4, 10)
Gradient sum ≈ 0 ? 0.0


# bloc GPT-2

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# GPT-2 Transformer Block pour NeuroDSL
# ══════════════════════════════════════════════════════════════════════════════

struct GPT2Block
    dim::Int
    n_heads::Int
    ff_mult::Int  # 4 pour GPT-2
end

function (b::GPT2Block)(g::NeuroDSL.NeuroGraph, x_sym::Symbol, prefix::Symbol; 
                        namespace=g.active_ns)
    ns = namespace
    d = b.dim
    
    # ── Pre-Attention RMSNorm ──
    ln1_sym = Symbol(prefix, :_ln1)
    gamma1 = Symbol(prefix, :_ln1_g)
    NeuroDSL.set!(g, gamma1, ones(Float32, d); is_param=true, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(ln1_sym, [x_sym, gamma1], :rmsnorm; namespace=ns))
    
    # ── Multi-Head Flash Attention ──
    attn_prefix = Symbol(prefix, :_attn)
    attn_out = MultiHeadFlashAttention(d, b.n_heads)(g, ln1_sym, attn_prefix; namespace=ns)
    
    # ── Residual 1 ──
    res1_sym = Symbol(prefix, :_res1)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(res1_sym, [x_sym, attn_out], :add; namespace=ns))
    
    # ── Pre-FFN RMSNorm ──
    ln2_sym = Symbol(prefix, :_ln2)
    gamma2 = Symbol(prefix, :_ln2_g)
    NeuroDSL.set!(g, gamma2, ones(Float32, d); is_param=true, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(ln2_sym, [res1_sym, gamma2], :rmsnorm; namespace=ns))
    
    # ── FeedForward (Linear → GELU → Linear) ──
    ff_hidden = d * b.ff_mult
    
    # Première projection (expand)
    ff1_out = Linear(d, ff_hidden, bias=true)(g, ln2_sym, Symbol(prefix, :_ff1); namespace=ns)
    
    # GELU (approximé par SwiGLU avec gate=up)
    gate_out = Linear(d, ff_hidden, bias=true)(g, ln2_sym, Symbol(prefix, :_gate); namespace=ns)
    act_sym = Symbol(prefix, :_act)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(act_sym, [gate_out, ff1_out], :swiglu; namespace=ns))
    
    # Deuxième projection (compress)
    ff2_out = Linear(ff_hidden, d, bias=true)(g, act_sym, Symbol(prefix, :_ff2); namespace=ns)
    
    # ── Residual 2 ──
    out_sym = Symbol(prefix, :_out)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [res1_sym, ff2_out], :add; namespace=ns))
    
    return out_sym
end

## TEST CPU 

In [4]:
using NeuroDSL

dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:test)

# Entrée : séquence de 8 tokens, dim=64
seq_len = 8
dim = 64
n_heads = 4

x = randn(Float32, seq_len, dim)
NeuroDSL.set!(g, :x, x; namespace=:test)

# Créer un bloc GPT-2
block = GPT2Block(dim, n_heads, 4)
out_sym = block(g, :x, :block1; namespace=:test)

# Forward
ctx = NeuroDSL.CtxStore()
NeuroDSL.demand!(g, out_sym; ctx_store=ctx, namespace=:test)
out_val = g.nodes[:test][out_sym].value
println("✅ Bloc GPT-2 forward OK")
println("   Entrée  : $(size(x))")
println("   Sortie  : $(size(out_val))")
println("   Nœuds   : $(length(g.nodes[:test]))")
println("   Règles  : $(length(g.rules[:test]))")

✅ Op :flash_attn enregistré
✅ Bloc GPT-2 forward OK
   Entrée  : (8, 64)
   Sortie  : (8, 64)
   Nœuds   : 42
   Règles  : 29


## TEST GPU 

In [5]:
using NeuroDSL, CUDA

dev = NeuroDSL.Backend.CUDADevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:test)

seq_len = 8
dim = 64
n_heads = 4

x = CUDA.randn(Float32, seq_len, dim)
NeuroDSL.set!(g, :x, x; namespace=:test)

block = GPT2Block(dim, n_heads, 4)
out_sym = block(g, :x, :block1; namespace=:test)

ctx = NeuroDSL.CtxStore()
NeuroDSL.demand!(g, out_sym; ctx_store=ctx, namespace=:test)
out_val = g.nodes[:test][out_sym].value
println("✅ Bloc GPT-2 forward OK")
println("   Entrée  : $(size(x))")
println("   Sortie  : $(size(out_val))")

✅ Bloc GPT-2 forward OK
   Entrée  : (8, 64)
   Sortie  : (8, 64)


# Assembler GPT-2 small complet

In [6]:
function build_gpt2_small(g::NeuroDSL.NeuroGraph; 
                          vocab_size=1000,  # Réduit pour test rapide
                          dim=256,          # Réduit pour test rapide
                          n_heads=8,
                          n_layers=12,
                          seq_len=32)
    ns = g.active_ns
    
    # ── Token Embedding ──
    set!(g, :wte, randn(Float32, vocab_size, dim) .* 0.02f0; is_param=true, namespace=ns)
    
    # ── Position Embedding ──
    set!(g, :wpe, randn(Float32, seq_len, dim) .* 0.02f0; is_param=true, namespace=ns)
    
    # ── Input ──
    set!(g, :input_ids, Int32.(rand(1:vocab_size, seq_len)); atom_type=NeuroDSL.Datom, namespace=ns)
    set!(g, :pos_ids, Int32.(collect(1:seq_len)); atom_type=NeuroDSL.Datom, namespace=ns)
        
    # Token emb
    tok_emb = :tok_emb
    addrule!(g, GraphRule(tok_emb, [:wte, :input_ids], :embedding; namespace=ns))
    
    # Position emb
    pos_emb = :pos_emb
    addrule!(g, GraphRule(pos_emb, [:wpe, :pos_ids], :embedding; namespace=ns))
    
    # Somme
    hidden = :hidden
    addrule!(g, GraphRule(hidden, [tok_emb, pos_emb], :add; namespace=ns))
    
    # ── 12 Transformer Blocks ──
    block = GPT2Block(dim, n_heads, 4)
    for i in 1:n_layers
        hidden = block(g, hidden, Symbol(:block_, i); namespace=ns)
    end
    
    # ── Final LayerNorm ──
    ln_final = :ln_final
    set!(g, :ln_final_g, ones(Float32, dim); is_param=true, namespace=ns)
    addrule!(g, GraphRule(ln_final, [hidden, :ln_final_g], :rmsnorm; namespace=ns))
    
    # ── LM Head ──
    logits = :logits
    set!(g, :lm_head, randn(Float32, vocab_size, dim) .* 0.02f0; is_param=true, namespace=ns)
    addrule!(g, GraphRule(logits, [ln_final, :lm_head], :matmul; 
             attrs=Dict(:trans_b=>true), namespace=ns))
    
    return logits
end

build_gpt2_small (generic function with 1 method)

## TEST GPT2-small

In [7]:
using NeuroDSL

dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:gpt2)

logits_sym = build_gpt2_small(g)

ctx = NeuroDSL.CtxStore()
NeuroDSL.demand!(g, logits_sym; ctx_store=ctx, namespace=:gpt2)
logits_val = g.nodes[:gpt2][logits_sym].value
println("✅ GPT-2 Small forward OK")
println("   Logits shape: $(size(logits_val))  # (seq_len, vocab_size)")
println("   Total nodes : $(length(g.nodes[:gpt2]))")
println("   Total rules : $(length(g.rules[:gpt2]))")

✅ GPT-2 Small forward OK
   Logits shape: (32, 1000)  # (seq_len, vocab_size)
   Total nodes : 695
   Total rules : 545


# Loss et le benchmark

In [8]:
using NeuroDSL

dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:gpt2)

logits_sym = build_gpt2_small(g)

# ─── Diagnostic : vérifier les types ───
println("\n─── Diagnostic des types ───")
for sym in [:input_ids, :pos_ids, :wte, :wpe]
    if haskey(g.nodes[:gpt2], sym)
        val = g.nodes[:gpt2][sym].value
        println("  $sym : $(typeof(val))  size=$(size(val))")
    end
end

ctx = NeuroDSL.CtxStore()
NeuroDSL.demand!(g, logits_sym; ctx_store=ctx, namespace=:gpt2)
logits_val = g.nodes[:gpt2][logits_sym].value
println("\n✅ GPT-2 Small forward OK")
println("   Logits shape: $(size(logits_val))")
println("   Total nodes : $(length(g.nodes[:gpt2]))")
println("   Total rules : $(length(g.rules[:gpt2]))")


─── Diagnostic des types ───
  input_ids : Vector{Float32}  size=(32,)
  pos_ids : Vector{Float32}  size=(32,)
  wte : Matrix{Float32}  size=(1000, 256)
  wpe : Matrix{Float32}  size=(32, 256)

✅ GPT-2 Small forward OK
   Logits shape: (32, 1000)
   Total nodes : 695
   Total rules : 545


In [10]:
using NeuroDSL

dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:gpt2)

logits_sym = build_gpt2_small(g)

# Diagnostic rapide
println("input_ids type: ", typeof(g.nodes[:gpt2][:input_ids].value))

# Forward
ctx = NeuroDSL.CtxStore()
NeuroDSL.demand!(g, logits_sym; ctx_store=ctx, namespace=:gpt2)
println("✅ Forward OK")

# Ajouter loss
labels = Float32.(rand(1:1000, 32))
NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=:gpt2)
NeuroDSL.set!(g, :loss, fill(0.0f0, 1, 1); namespace=:gpt2)
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits_sym, :labels], :cross_entropy; namespace=:gpt2))

NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
println("Loss = ", g.nodes[:gpt2][:loss].value[1])

NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2)
println("✅ Backward OK")

input_ids type: Vector{Float32}
✅ Forward OK
Loss = 6.939813
✅ Backward OK


In [11]:
# ─── Geler les 10 premières couches ───
using Statistics
frozen_count = 0
trainable_count = 0

for i in 1:10
    prefix = string(:block_, i)
    for (sym, nd) in g.nodes[:gpt2]
        sym_str = string(sym)
        if startswith(sym_str, prefix) && length(sym_str) > length(prefix) && sym_str[length(prefix)+1] == '_'
            if nd.is_param
                nd.is_param = false
                frozen_count += 1
            end
        end
    end
end

# Compter
trainable = 0
frozen = 0
for (_, nd) in g.nodes[:gpt2]
    if nd.is_param == true
        trainable += 1
    elseif nd.is_param == false && nd.atom_type <: NeuroDSL.Quantom
        frozen += 1
    end
end

println("\n=== Configuration Fine-Tuning ===")
println("  Paramètres gelés        : $frozen")
println("  Paramètres entraînables : $trainable")
println("  Ratio entraînable       : $(round(trainable/(trainable+frozen)*100, digits=1))%")

# ─── Warm-up ───
for _ in 1:3
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, sparse=true)
end

# ─── Benchmark sparse (50 itérations) ───
times = Float64[]
for _ in 1:50
    t = @elapsed begin
        NeuroDSL.invalidate_all!(g; namespace=:gpt2)
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, sparse=true)
    end
    push!(times, t * 1000)
end

println("\n=== Résultats GPT-2 Small — CPU (backward sparse, 10/12 couches gelées) ===")
println("  Temps médian : $(round(median(times), digits=1)) ms")
println("  ± écart-type : $(round(std(times), digits=1)) ms")
println("  Min / Max    : $(round(minimum(times), digits=1)) / $(round(maximum(times), digits=1)) ms")



=== Configuration Fine-Tuning ===
  Paramètres gelés        : 666
  Paramètres entraînables : 28
  Ratio entraînable       : 4.0%

=== Résultats GPT-2 Small — CPU (backward sparse, 10/12 couches gelées) ===
  Temps médian : 897.0 ms
  ± écart-type : 42.8 ms
  Min / Max    : 804.4 / 1064.1 ms


In [21]:
# ─── Benchmark SANS backward sparse ───
times_full = Float64[]
for _ in 1:20  # 20 itérations suffisent pour comparer
    t = @elapsed begin
        NeuroDSL.invalidate_all!(g; namespace=:gpt2)
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, full=true)  # ← standard
    end
    push!(times_full, t * 1000)
end

println("\n=== Comparaison Backward ===")
println("  Backward standard   : $(round(median(times_full), digits=1)) ms")
println("  Backward sparse     : $(round(median(times), digits=1)) ms")
gain = (median(times_full) - median(times)) / median(times_full) * 100
println("  Gain                : $(round(gain, digits=1))%")


=== Comparaison Backward ===
  Backward standard   : 497.1 ms
  Backward sparse     : 897.0 ms
  Gain                : -80.4%


In [30]:
# ═══ Test Mémoire CPU — Backward Sparse vs Standard ═══

dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.NeuroGraph(device=dev, namespace=:gpt2)

logits_sym = build_gpt2_small(g)

# Labels et loss
labels = Float32.(rand(1:1000, 32))
NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=:gpt2)
NeuroDSL.set!(g, :loss, fill(0.0f0, 1, 1); namespace=:gpt2)
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits_sym, :labels], :cross_entropy; namespace=:gpt2))

# Geler les 10 premières couches
for i in 1:10
    prefix = string(:block_, i)
    for (sym, nd) in g.nodes[:gpt2]
        sym_str = string(sym)
        if startswith(sym_str, prefix) && length(sym_str) > length(prefix) && sym_str[length(prefix)+1] == '_'
            if nd.is_param
                nd.is_param = false
            end
        end
    end
end

ctx = NeuroDSL.CtxStore()

# Warm-up
for _ in 1:2
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2)
end
GC.gc()

# ─── Mesure : Backward STANDARD ───
GC.gc()
NeuroDSL.invalidate_all!(g; namespace=:gpt2)
NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)

allocs_full = @allocated begin
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, full=true)
end

# Compter les gradients stockés
grads_full = 0
for (_, nd) in g.nodes[:gpt2]
    if nd.gradient !== nothing
        grads_full += 1
    end
end

# ─── Mesure : Backward SPARSE ───
GC.gc()
NeuroDSL.invalidate_all!(g; namespace=:gpt2)
NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)

allocs_sparse = @allocated begin
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, sparse=true)
end

grads_sparse = 0
for (_, nd) in g.nodes[:gpt2]
    if nd.gradient !== nothing
        grads_sparse += 1
    end
end

# ─── Résultats ───
println("\n=== Mémoire CPU — GPT-2 Small ===")
println("  Allocations (standard) : $(round(allocs_full / 1024, digits=1)) KB")
println("  Allocations (sparse)   : $(round(allocs_sparse / 1024, digits=1)) KB")
println("  Différence             : $(round((allocs_full - allocs_sparse) / 1024, digits=1)) KB")
println()
println("  Gradients stockés (standard) : $grads_full")
println("  Gradients stockés (sparse)   : $grads_sparse")


=== Mémoire CPU — GPT-2 Small ===
  Allocations (standard) : 139288.3 KB
  Allocations (sparse)   : 87888.9 KB
  Différence             : 51399.5 KB

  Gradients stockés (standard) : 148
  Gradients stockés (sparse)   : 28


In [33]:
using CUDA

println("=== Backward SPARSE ===")
CUDA.@time begin
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, sparse=true)
    CUDA.synchronize()
end

println("\n=== Backward STANDARD ===")
CUDA.@time begin
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, full=true)
    CUDA.synchronize()
end

=== Backward SPARSE ===
  0.787319 seconds (23.75 k CPU allocations: 92.490 MiB)

=== Backward STANDARD ===
  0.501547 seconds (26.21 k CPU allocations: 142.685 MiB)


In [35]:
using CUDA

println("=== Backward SPARSE ===")
# Warm‑up (non mesuré)
NeuroDSL.invalidate_all!(g; namespace=:gpt2)
NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, sparse=true)
CUDA.synchronize()

CUDA.@time begin
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, sparse=true)
    CUDA.synchronize()
end

println("\n=== Backward STANDARD ===")
# Warm‑up (non mesuré)
NeuroDSL.invalidate_all!(g; namespace=:gpt2)
NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, full=true)
CUDA.synchronize()

CUDA.@time begin
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:gpt2, full=true)
    CUDA.synchronize()
end

=== Backward SPARSE ===
  0.820434 seconds (23.75 k CPU allocations: 92.490 MiB)

=== Backward STANDARD ===
  0.759450 seconds (26.21 k CPU allocations: 142.685 MiB, 39.66% gc time)


# Forward Incrémental sur GPT-2 avec VRAI Dataset

In [15]:
using NeuroDSL, Statistics

# ═══ GPT-2 Small avec Wikitext-2 ═══
# On utilise des données aléatoires pour le benchmark
# (Wikitext nécessite un tokenizer, on simplifie)

function benchmark_forward_incremental_gpt2()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.NeuroGraph(device=dev, namespace=:gpt2)
    
    # Construire GPT-2 small
    logits_sym = build_gpt2_small(g; vocab_size=1000, dim=256, n_layers=12, seq_len=32)
    
    # Labels aléatoires (simule un vrai dataset)
    labels = Float32.(rand(1:1000, 32))
    NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=:gpt2)
    NeuroDSL.set!(g, :loss, fill(0.0f0, 1, 1); namespace=:gpt2)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits_sym, :labels], :cross_entropy; namespace=:gpt2))
    
    # Geler les 10 premières couches (fine-tuning des 2 dernières)
    for i in 1:10
        prefix = string(:block_, i)
        for (sym, nd) in g.nodes[:gpt2]
            sym_str = string(sym)
            if startswith(sym_str, prefix) && length(sym_str) > length(prefix) && sym_str[length(prefix)+1] == '_'
                if nd.is_param
                    nd.is_param = false
                end
            end
        end
    end
    
    ctx = NeuroDSL.CtxStore()
    
    # ═══ Mesure : Forward COMPLET ═══
    println("Mesure forward complet...")
    times_full = Float64[]
    for _ in 1:30
        NeuroDSL.invalidate_all!(g; namespace=:gpt2)
        t = @elapsed NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
        push!(times_full, t * 1000)
    end
    t_full = median(times_full)
    
    # ═══ Mesure : Forward INCRÉMENTAL ═══
    println("Mesure forward incrémental...")
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)  # État valide
    
    times_incr = Float64[]
    for _ in 1:30
        # Modifier UNIQUEMENT les labels (simule un nouveau batch)
        new_labels = Float32.(rand(1:1000, 32))
        g.nodes[:gpt2][:labels].value .= new_labels
        NeuroDSL._invalidate_downstream!(g, :labels, :gpt2)
        
        t = @elapsed NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
        push!(times_incr, t * 1000)
    end
    t_incr = median(times_incr)
    
    # ═══ Résultats ═══
    gain = (t_full - t_incr) / t_full * 100
    println("\n=== Forward Incrémental — GPT-2 Small Fine-Tuning ===")
    println("  Forward complet      : $(round(t_full, digits=1)) ms")
    println("  Forward incrémental  : $(round(t_incr, digits=1)) ms")
    println("  Gain                 : $(round(gain, digits=1))%")
    println("  Couches recalculées  : 2/12 (seulement les couches fine-tunées)")
    println("  Couches préservées   : 10/12 (backbone gelé)")
    
    return t_full, t_incr
end

t_full, t_incr = benchmark_forward_incremental_gpt2()

Mesure forward complet...
Mesure forward incrémental...

=== Forward Incrémental — GPT-2 Small Fine-Tuning ===
  Forward complet      : 166.8 ms
  Forward incrémental  : 0.6 ms
  Gain                 : 99.7%
  Couches recalculées  : 2/12 (seulement les couches fine-tunées)
  Couches préservées   : 10/12 (backbone gelé)


(166.76549999999997, 0.55345)

In [38]:
using NeuroDSL, Random

function benchmark_forward_incremental_realistic()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.NeuroGraph(device=dev, namespace=:gpt2)
    
    # Construire GPT-2 small
    logits_sym = build_gpt2_small(g; vocab_size=1000, dim=256, n_layers=12, seq_len=32)
    
    # Labels
    labels = Float32.(rand(1:1000, 32))
    NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=:gpt2)
    NeuroDSL.set!(g, :loss, fill(0.0f0, 1, 1); namespace=:gpt2)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits_sym, :labels], :cross_entropy; namespace=:gpt2))
    
    # Geler les 10 premières couches
    for i in 1:10
        prefix = string(:block_, i)
        for (sym, nd) in g.nodes[:gpt2]
            sym_str = string(sym)
            if startswith(sym_str, prefix) && length(sym_str) > length(prefix) && sym_str[length(prefix)+1] == '_'
                if nd.is_param
                    nd.is_param = false
                end
            end
        end
    end
    
    ctx = NeuroDSL.CtxStore()
    
    # ═══ Forward COMPLET ═══
    println("Mesure forward complet...")
    times_full = Float64[]
    for _ in 1:30
        # Changer les poids des couches fine-tunées (simule un pas d'optimiseur)
        for (sym, nd) in g.nodes[:gpt2]
            if nd.is_param  # Seulement les couches 11-12 + lm_head
                nd.value .+= randn(Float32, size(nd.value)...) .* 0.001f0
            end
        end
        NeuroDSL.invalidate_all!(g; namespace=:gpt2)
        t = @elapsed NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
        push!(times_full, t * 1000)
    end
    t_full = median(times_full)
    
    # ═══ Forward INCRÉMENTAL ═══
    println("Mesure forward incrémental...")
    # Remettre le graphe en état valide
    NeuroDSL.invalidate_all!(g; namespace=:gpt2)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
    
    times_incr = Float64[]
    for _ in 1:30
        for (sym, nd) in g.nodes[:gpt2]
            if nd.is_param
                nd.value .+= randn(Float32, size(nd.value)...) .* 0.001f0
            end
        end
        NeuroDSL._invalidate_downstream!(g, :block_10_out, :gpt2)  # un seul BFS
        t = @elapsed NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:gpt2)
        push!(times_incr, t * 1000)
    end
    t_incr = median(times_incr)
    
    # ═══ Compter les couches recalculées ═══
    n_recomputed = 0
    n_preserved = 0
    for (sym, nd) in g.nodes[:gpt2]
        if haskey(g.rules[:gpt2], sym)
            if nd.valid
                n_preserved += 1
            end
        end
    end
    
    # ═══ Résultats ═══
    gain = (t_full - t_incr) / t_full * 100
    println("\n=== Forward Incrémental — GPT-2 Small (après pas d'optimiseur) ===")
    println("  Forward complet      : $(round(t_full, digits=1)) ms")
    println("  Forward incrémental  : $(round(t_incr, digits=1)) ms")
    println("  Gain                 : $(round(gain, digits=1))%")
    println("  Speedup              : $(round(t_full / t_incr, digits=1))×")
    println("  Nœuds préservés      : $n_preserved / $(length(g.nodes[:gpt2]))")
    
    return t_full, t_incr, gain
end

t_full, t_incr, gain = benchmark_forward_incremental_realistic()

Mesure forward complet...
Mesure forward incrémental...

=== Forward Incrémental — GPT-2 Small (après pas d'optimiseur) ===
  Forward complet      : 158.7 ms
  Forward incrémental  : 29.7 ms
  Gain                 : 81.3%
  Speedup              : 5.3×
  Nœuds préservés      : 546 / 697


(158.65235, 29.7165, 81.26942336498641)

# Multi Tête 

In [17]:
using NeuroDSL, Random

function benchmark_multi_head_finetune()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.NeuroGraph(device=dev, namespace=:multi)
    
    # ═══ Backbone GPT-2 Small (10 couches gelées) ═══
    # On récupère la sortie DU BACKBONE (avant LM head)
    logits_sym = build_gpt2_small(g; vocab_size=1000, dim=256, n_layers=12, seq_len=32)
    
    # Le backbone s'arrête à :ln_final (juste avant :lm_head)
    backbone_output = :ln_final  # Sortie du backbone (32, 256)
    
    # Geler les 10 premières couches + lm_head
    for i in 1:10
        prefix = string(:block_, i)
        for (sym, nd) in g.nodes[:multi]
            sym_str = string(sym)
            if startswith(sym_str, prefix) && length(sym_str) > length(prefix) && sym_str[length(prefix)+1] == '_'
                if nd.is_param
                    nd.is_param = false
                end
            end
        end
    end
    # Geler aussi le lm_head (on ne fine-tune que les têtes)
    if haskey(g.nodes[:multi], :lm_head)
        g.nodes[:multi][:lm_head].is_param = false
    end
    
    # ═══ 3 Têtes de Fine-Tuning (branchées sur la sortie du backbone) ═══
    hidden_dim = 128
    n_heads = 3
    
    for h in 1:n_heads
        prefix = Symbol(:head_, h)
        
        # Couche cachée : Linear(256 → hidden_dim) + ReLU
        w1 = Symbol(prefix, :_w1)
        b1 = Symbol(prefix, :_b1)
        h1 = Symbol(prefix, :_h1)
        NeuroDSL.set!(g, w1, randn(Float32, hidden_dim, 256) .* 0.02f0; is_param=true, namespace=:multi)
        NeuroDSL.set!(g, b1, randn(Float32, hidden_dim) .* 0.02f0; is_param=true, namespace=:multi)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(h1, [backbone_output, w1, b1], :linear; namespace=:multi))
        
        # Activation
        a1 = Symbol(prefix, :_a1)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(a1, [h1], :relu; namespace=:multi))
        
        # Couche de sortie : Linear(hidden_dim → out_dim)
        w2 = Symbol(prefix, :_w2)
        b2 = Symbol(prefix, :_b2)
        out_sym = Symbol(prefix, :_out)
        out_dim = (h == 1) ? 2 : 1000
        NeuroDSL.set!(g, w2, randn(Float32, out_dim, hidden_dim) .* 0.02f0; is_param=true, namespace=:multi)
        NeuroDSL.set!(g, b2, randn(Float32, out_dim) .* 0.02f0; is_param=true, namespace=:multi)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [a1, w2, b2], :linear; namespace=:multi))
        
        # Labels et loss
        label_sym = Symbol(prefix, :_labels)
        loss_sym = Symbol(prefix, :_loss)
        NeuroDSL.set!(g, label_sym, Float32.(rand(1:out_dim, 32)); atom_type=NeuroDSL.Datom, namespace=:multi)
        NeuroDSL.set!(g, loss_sym, fill(0.0f0, 1, 1); namespace=:multi)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [out_sym, label_sym], :cross_entropy; namespace=:multi))
    end
    
    ctx = NeuroDSL.CtxStore()
    
    # ═══ Warm-up ═══
    for h in 1:n_heads
        loss_sym = Symbol(:head_, h, :_loss)
        NeuroDSL.invalidate_all!(g; namespace=:multi)
        NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=:multi)
    end
    
    # ═══ Forward COMPLET (3 têtes) ═══
    println("Mesure forward complet (3 têtes)...")
    times_full = Float64[]
    for _ in 1:20
        NeuroDSL.invalidate_all!(g; namespace=:multi)
        t = @elapsed begin
            for h in 1:n_heads
                NeuroDSL.demand!(g, Symbol(:head_, h, :_loss); ctx_store=ctx, namespace=:multi)
            end
        end
        push!(times_full, t * 1000)
    end
    t_full = median(times_full)
    
    # ═══ Forward INCRÉMENTAL (1 tête modifiée) ═══
    println("Mesure forward incrémental (1 tête modifiée)...")
    NeuroDSL.invalidate_all!(g; namespace=:multi)
    for h in 1:n_heads
        NeuroDSL.demand!(g, Symbol(:head_, h, :_loss); ctx_store=ctx, namespace=:multi)
    end
    
    times_incr = Float64[]
    for _ in 1:20
        # Modifier UNIQUEMENT les poids de la Tête A
        for (sym, nd) in g.nodes[:multi]
            sym_str = string(sym)
            if startswith(sym_str, "head_1") && nd.is_param
                nd.value .+= randn(Float32, size(nd.value)...) .* 0.001f0
                NeuroDSL._invalidate_downstream!(g, sym, :multi)
            end
        end
        
        t = @elapsed NeuroDSL.demand!(g, :head_1_loss; ctx_store=ctx, namespace=:multi)
        push!(times_incr, t * 1000)
    end
    t_incr = median(times_incr)
    
    # ═══ Résultats ═══
    gain = (t_full - t_incr) / t_full * 100
    println("\n=== Forward Incrémental — Multi-Têtes ===")
    println("  Forward complet (3 têtes)   : $(round(t_full, digits=1)) ms")
    println("  Forward incrémental (1 tête) : $(round(t_incr, digits=1)) ms")
    println("  Gain                        : $(round(gain, digits=1))%")
    println("  Speedup                     : $(round(t_full / t_incr, digits=1))×")
    
    return t_full, t_incr, gain
end

t_full, t_incr, gain = benchmark_multi_head_finetune()

Mesure forward complet (3 têtes)...
Mesure forward incrémental (1 tête modifiée)...

=== Forward Incrémental — Multi-Têtes ===
  Forward complet (3 têtes)   : 171.6 ms
  Forward incrémental (1 tête) : 1.6 ms
  Gain                        : 99.0%
  Speedup                     : 105.0×


(171.61554999999998, 1.63465, 99.04749307390853)

# Nas Dynamique 


In [26]:
using NeuroDSL, Random

function benchmark_nas_dynamic()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.NeuroGraph(device=dev, namespace=:nas)
    
    # Dataset fixe
    x = randn(Float32, 64, 128)
    y = randn(Float32, 64, 10)
    NeuroDSL.set!(g, :x, x; namespace=:nas)
    NeuroDSL.set!(g, :y, y; atom_type=NeuroDSL.Datom, namespace=:nas)
    
    # Première couche (fixe)
    NeuroDSL.set!(g, :W0, randn(Float32, 256, 128) .* 0.02f0; is_param=true, namespace=:nas)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:h0, [:x, :W0], :matmul; attrs=Dict(:trans_b=>true), namespace=:nas))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:a0, [:h0], :relu; namespace=:nas))
    
    prev = :a0
    current_layers = []
    
    # ═══ Ajouter 20 architectures différentes ═══
    n_architectures = 20
    times = Float64[]
    ctx = NeuroDSL.CtxStore()
    
    for arch in 1:n_architectures
        # Paramètres aléatoires pour cette architecture
        n_new_layers = rand(1:3)  # 1 à 3 nouvelles couches
        hidden_dim = rand([64, 128, 256])  # Largeur variable
        
        # Ajouter les couches
        new_syms = []
        for i in 1:n_new_layers
            w_sym = Symbol(:W_arch, arch, :_, i)
            h_sym = Symbol(:h_arch, arch, :_, i)
            a_sym = Symbol(:a_arch, arch, :_, i)
            out_dim = (i == n_new_layers) ? 10 : hidden_dim
            in_dim = (i == 1) ? 256 : hidden_dim
            
            NeuroDSL.set!(g, w_sym, randn(Float32, out_dim, in_dim) .* 0.02f0; is_param=true, namespace=:nas)
            NeuroDSL.addrule!(g, NeuroDSL.GraphRule(h_sym, [prev, w_sym], :matmul; attrs=Dict(:trans_b=>true), namespace=:nas))
            NeuroDSL.addrule!(g, NeuroDSL.GraphRule(a_sym, [h_sym], :relu; namespace=:nas))
            push!(new_syms, w_sym, h_sym, a_sym)
            prev = a_sym
        end
        
        # Loss
        loss_sym = Symbol(:loss_arch, arch)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :y], :mse_loss; namespace=:nas))
        
        # Mesurer le temps d'évaluation
        t = @elapsed NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=:nas)
        push!(times, t * 1000)
        
        # Supprimer les couches ajoutées (pour la prochaine architecture)
        for sym in new_syms
            delete!(g.rules[:nas], sym)
            delete!(g.nodes[:nas], sym)
        end
        delete!(g.rules[:nas], loss_sym)
        delete!(g.nodes[:nas], loss_sym)
        g._topo_cache[:nas] = nothing
        
        # Remettre prev à la sortie de la couche 0
        prev = :a0
    end
    
    # ═══ Résultats ═══
    println("\n=== NAS Dynamique — $n_architectures Architectures ===")
    println("  Temps médian par architecture : $(round(median(times), digits=1)) ms")
    println("  Temps total                   : $(round(sum(times), digits=1)) ms")
    println("  Architectures explorées       : $n_architectures")
    
    return times
end

times_nas = benchmark_nas_dynamic()


=== NAS Dynamique — 20 Architectures ===
  Temps médian par architecture : 1.4 ms
  Temps total                   : 28.5 ms
  Architectures explorées       : 20


20-element Vector{Float64}:
 3.8177
 3.8968000000000003
 2.6284
 1.4938
 0.12860000000000002
 2.1504
 1.2348000000000001
 1.3229
 1.5472000000000001
 0.1396
 1.9622
 1.582
 1.0459
 0.1237
 0.1186
 0.1439
 1.3146
 0.12449999999999999
 2.0712
 1.6878

# Continual learning 

In [19]:
using NeuroDSL, Random

function benchmark_continual_learning()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.NeuroGraph(device=dev, namespace=:cl)

    D = 64
    batch = 32
    lr = 1f-3  # SGD learning rate

    # Données (3 tâches différentes)
    X1 = randn(Float32, batch, D)
    Y1 = reshape(Float32.(X1[:, 1] .> 0), batch, 1)   # (batch, 1)
    X2 = randn(Float32, batch, D) .+ 2.0f0
    Y2 = reshape(Float32.(X2[:, 2] .> 1), batch, 1)
    X3 = randn(Float32, batch, D) .- 1.0f0
    Y3 = reshape(Float32.(X3[:, 3] .< 0), batch, 1)

    # Réseau : x → W1 → W2 → W3 → loss
    NeuroDSL.set!(g, :W1, randn(Float32, D, D) .* 0.1f0; is_param=true, namespace=:cl)
    NeuroDSL.set!(g, :W2, randn(Float32, D, D) .* 0.1f0; is_param=true, namespace=:cl)
    NeuroDSL.set!(g, :W3, randn(Float32, 1, D)  .* 0.1f0; is_param=true, namespace=:cl)

    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:h1, [:x, :W1], :matmul; attrs=Dict(:trans_b=>true), namespace=:cl))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:h2, [:h1, :W2], :matmul; attrs=Dict(:trans_b=>true), namespace=:cl))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:h3, [:h2, :W3], :matmul; attrs=Dict(:trans_b=>true), namespace=:cl))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [:h3, :y], :mse_loss; namespace=:cl))
    NeuroDSL.set!(g, :y, Y1; atom_type=NeuroDSL.Datom, namespace=:cl)

    ctx = NeuroDSL.CtxStore()

    # ─── Helper SGD step ───
    function sgd_step!(g, ns)
        for (_, nd) in g.nodes[ns]
            if nd.is_param && nd.gradient !== nothing
                nd.value .-= lr .* nd.gradient
            end
        end
    end

    # ═══ Tâche 1 : Tout entraîner (W1, W2, W3) ═══
    println("Tâche 1 — Entraînement complet (W1+W2+W3)...")
    NeuroDSL.set!(g, :x, X1; namespace=:cl)
    NeuroDSL.set!(g, :y, Y1; atom_type=NeuroDSL.Datom, namespace=:cl)
    for step in 1:100
        NeuroDSL.invalidate_all!(g; namespace=:cl)
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:cl)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:cl)
        sgd_step!(g, :cl)
    end
    loss_t1 = g.nodes[:cl][:loss].value[1]
    println("  Loss finale Tâche 1 : $(round(loss_t1, digits=4))")

    # Geler W1 et W3 — les moments Adam de W2 sont préservés
    g.nodes[:cl][:W1].is_param = false
    g.nodes[:cl][:W3].is_param = false

    # ═══ Tâche 2 : Seul W2 entraînable ═══
    println("Tâche 2 — Seul W2 entraînable (W1, W3 gelés)...")
    NeuroDSL.set!(g, :x, X2; namespace=:cl)
    NeuroDSL.set!(g, :y, Y2; atom_type=NeuroDSL.Datom, namespace=:cl)
    for step in 1:100
        NeuroDSL.invalidate_all!(g; namespace=:cl)
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:cl)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:cl, sparse=true)
        sgd_step!(g, :cl)  # ne touche que W2 (is_param=true)
    end
    loss_t2 = g.nodes[:cl][:loss].value[1]
    println("  Loss finale Tâche 2 : $(round(loss_t2, digits=4))")
    println("  W2.grad : ", g.nodes[:cl][:W2].gradient !== nothing ? "PRÉSENT ✅" : "RIEN ❌")

    # ═══ Tâche 3 : W1 et W3 réactivés, W2 gelé ═══
    g.nodes[:cl][:W1].is_param = true
    g.nodes[:cl][:W2].is_param = false
    g.nodes[:cl][:W3].is_param = true

    println("Tâche 3 — W1, W3 réactivés (W2 gelé)...")
    NeuroDSL.set!(g, :x, X3; namespace=:cl)
    NeuroDSL.set!(g, :y, Y3; atom_type=NeuroDSL.Datom, namespace=:cl)
    for step in 1:100
        NeuroDSL.invalidate_all!(g; namespace=:cl)
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:cl)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:cl, sparse=true)
        sgd_step!(g, :cl)  # ne touche que W1 et W3
    end
    loss_t3 = g.nodes[:cl][:loss].value[1]
    println("  Loss finale Tâche 3 : $(round(loss_t3, digits=4))")

    # ─── Vérification des gradients ───
    println("\n=== Vérification backward sparse ===")
    println("  W1.grad : ", g.nodes[:cl][:W1].gradient !== nothing ? "PRÉSENT ✅" : "RIEN ❌")
    println("  W2.grad : ", g.nodes[:cl][:W2].gradient === nothing ? "RIEN ✅ (gelé)"  : "PRÉSENT ❌")
    println("  W3.grad : ", g.nodes[:cl][:W3].gradient !== nothing ? "PRÉSENT ✅" : "RIEN ❌")
end

benchmark_continual_learning()


Tâche 1 — Entraînement complet (W1+W2+W3)...
  Loss finale Tâche 1 : 0.3704
Tâche 2 — Seul W2 entraînable (W1, W3 gelés)...
  Loss finale Tâche 2 : 0.1402
  W2.grad : PRÉSENT ✅
Tâche 3 — W1, W3 réactivés (W2 gelé)...
  Loss finale Tâche 3 : 0.0907

=== Vérification backward sparse ===
  W1.grad : PRÉSENT ✅
  W2.grad : RIEN ✅ (gelé)
  W3.grad : PRÉSENT ✅


In [20]:
using LinearAlgebra
function benchmark_gradient_inspection()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.NeuroGraph(device=dev, namespace=:debug)
    
    # Réseau avec 5 couches
    D = 128
    NeuroDSL.set!(g, :x, randn(Float32, 32, D); namespace=:debug)
    prev = :x
    for i in 1:5
        w = Symbol(:W, i)
        h = Symbol(:h, i)
        NeuroDSL.set!(g, w, randn(Float32, D, D).*0.1f0; is_param=true, namespace=:debug)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(h, [prev, w], :matmul; 
            attrs=Dict(:trans_b=>true), namespace=:debug))
        prev = h
    end
    NeuroDSL.set!(g, :y, randn(Float32, 32, D); atom_type=NeuroDSL.Datom, namespace=:debug)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [prev, :y], :mse_loss; namespace=:debug))
    
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:debug)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=:debug)
    
    # ═══ Inspection immédiate de tous les gradients ═══
    println("=== Inspection des gradients (sans réexécution) ===")
    for i in 1:5
        w = Symbol(:W, i)
        h = Symbol(:h, i)
        grad_norm = g.nodes[:debug][w].gradient !== nothing ? 
                    round(norm(g.nodes[:debug][w].gradient), digits=4) : "RIEN"
        val_norm = round(norm(g.nodes[:debug][h].value), digits=2)
        println("  Couche $i : ‖h‖=$(val_norm), ‖grad W‖=$(grad_norm)")
    end
    
    # ═══ Modifier UNE couche et réinspecter ═══
    println("\n=== Modification de W3 ===")
    g.nodes[:debug][Symbol(:W, 3)].value .*= 2.0f0
    NeuroDSL._invalidate_downstream!(g, Symbol(:W, 3), :debug)
    
    # Inspection AVANT recalcul
    println("Avant recalcul :")
    for i in 3:5
        h = Symbol(:h, i)
        valid_str = g.nodes[:debug][h].valid ? "valide ✅" : "invalide ⚠️"
        println("  h$i : $valid_str")
    end
    
    # Recalcul incrémental
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=:debug)
    
    println("Après recalcul incrémental :")
    for i in 3:5
        h = Symbol(:h, i)
        valid_str = g.nodes[:debug][h].valid ? "valide ✅" : "invalide ⚠️"
        println("  h$i : $valid_str")
    end
end

benchmark_gradient_inspection()

=== Inspection des gradients (sans réexécution) ===
  Couche 1 : ‖h‖=73.93, ‖grad W‖=2.9714
  Couche 2 : ‖h‖=83.07, ‖grad W‖=2.8958
  Couche 3 : ‖h‖=94.55, ‖grad W‖=2.652
  Couche 4 : ‖h‖=107.79, ‖grad W‖=2.3456
  Couche 5 : ‖h‖=121.53, ‖grad W‖=1.8681

=== Modification de W3 ===
Avant recalcul :
  h3 : invalide ⚠️
  h4 : invalide ⚠️
  h5 : invalide ⚠️
Après recalcul incrémental :
  h3 : valide ✅
  h4 : valide ✅
  h5 : valide ✅
